# Email Enrichment Pipeline — Provider Test Notebook

**Goal:** Test each free-tier provider one-by-one to validate:
1. Can we get director names from CIN? (InstaFinancials)
2. Can we get emails from name + domain? (Hunter, Snov.io, Prospeo)
3. Can we browse/search companies for contacts? (Apollo, Lusha, Lix)

**Instructions:** Fill in your API keys in the config cell below after signing up for free tiers.

---

## 0. Setup & Config

In [ ]:
# ============================================================
# CONFIG — Fill in your API keys after signing up for free tiers
# ============================================================

CONFIG = {
    # 1. InstaFinancials — Free API key
    #    Sign up: https://www.instafinancials.com → Register → Get free API key
    #    Docs: https://api.instafinancials.com/Docs
    #    Free tier: Unlimited InstaBasic lookups (company master + directors)
    "INSTAFINANCIALS_API_KEY": "",
    "INSTAFINANCIALS_BASE_URL": "https://api.instafinancials.com",

    # 2. Hunter.io — Free API key
    #    Sign up: https://hunter.io/users/sign_up
    #    Free tier: 25 searches + 50 verifications/month (renews on signup anniversary)
    #    API available on free tier: YES
    "HUNTER_API_KEY": "",

    # 3. Snov.io — Free API key
    #    Sign up: https://app.snov.io/register
    #    Free tier: 50 credits/month (renews monthly)
    #    API available on free tier: YES (limited)
    "SNOV_CLIENT_ID": "",
    "SNOV_CLIENT_SECRET": "",

    # 4. Prospeo — Free tier
    #    Sign up: https://prospeo.io/
    #    Free tier: 75 emails/month + 100 Chrome ext credits
    #    API available on free tier: UNCLEAR — test below
    "PROSPEO_API_KEY": "",

    # 5. Apollo.io — Free tier
    #    Sign up: https://www.apollo.io/sign-up (USE CORPORATE EMAIL for 10k credits)
    #    Free tier: Unlimited email views, 10 exports/month
    #    API available on free tier: NO (view-only via UI)
    #    We'll test their search API anyway to see what's possible
    "APOLLO_API_KEY": "",

    # 6. Lusha — Free tier
    #    Sign up: https://www.lusha.com/
    #    Free tier: 40-70 credits/month (renews on signup anniversary)
    #    API available on free tier: NO (Scale plan only)
    #    Usage: Chrome extension + web app only
    #    Keeping placeholder for future if we upgrade
    "LUSHA_API_KEY": "",

    # 7. Lix — Free tier
    #    Sign up: https://lix-it.com/
    #    Free tier: 50 verified emails/month
    #    API available: They have an API but unclear if free tier includes it
    "LIX_API_KEY": "",
}

# Test CINs — use a few known companies to validate
TEST_CINS = [
    "U72200KA2012PTC066107",  # Flipkart Internet Private Limited
    "U74999HR2011PTC043327",  # Zomato Private Limited (old CIN)
    "U72501KA2016PTC092387",  # InstaFinancials own CIN (from their docs)
]

# Test domains — for email pattern discovery
TEST_DOMAINS = [
    "flipkart.com",
    "zomato.com",
    "instafinancials.com",
]

print("Config loaded. Fill in API keys above before running provider tests.")

In [ ]:
import requests
import json
import time
from pprint import pprint

def safe_request(method, url, **kwargs):
    """Wrapper with error handling and rate-limit respect."""
    try:
        resp = requests.request(method, url, timeout=30, **kwargs)
        print(f"  → {resp.status_code} {resp.reason}")
        if resp.status_code == 429:
            print("  ⚠️  Rate limited! Wait and retry.")
            return None
        if resp.status_code >= 400:
            print(f"  ❌ Error: {resp.text[:500]}")
            return None
        return resp.json() if resp.headers.get('content-type', '').startswith('application/json') else resp.text
    except Exception as e:
        print(f"  ❌ Exception: {e}")
        return None

print("Utilities loaded.")

---
## 1. InstaFinancials — CIN → Director Names (FREE, Unlimited)

**What we're testing:**
- Input: CIN
- Expected output: Company name, director names, DINs, registered address
- API endpoint: `/InstaBasic/V1/json/CompanyCIN/{CIN}/all`
- Auth: `user-key` header

**Sign up at:** https://api.instafinancials.com/Docs → Click 'Get Free API Key'

In [ ]:
def instafinancials_get_company(cin):
    """
    InstaFinancials InstaBasic API — Get company master data + directors by CIN.
    
    FREE tier: Unlimited lookups.
    Returns: company_name, status, address, directors[{name, din, designation}]
    Does NOT return: email, phone, domain
    """
    api_key = CONFIG["INSTAFINANCIALS_API_KEY"]
    if not api_key:
        print("❌ INSTAFINANCIALS_API_KEY not set. Sign up at https://api.instafinancials.com/Docs")
        return None
    
    base_url = CONFIG["INSTAFINANCIALS_BASE_URL"]
    url = f"{base_url}/InstaBasic/V1/json/CompanyCIN/{cin}/all"
    
    print(f"\n🔍 InstaFinancials: Looking up CIN = {cin}")
    data = safe_request("GET", url, headers={"user-key": api_key})
    
    if data is None:
        return None
    
    # Parse the response — structure may vary, inspect and adapt
    print("\n📋 Raw response keys:")
    if isinstance(data, dict):
        print(list(data.keys()))
    
    return data


def extract_directors(insta_data):
    """
    Extract director names and DINs from InstaFinancials response.
    Adapt the key paths based on actual API response structure.
    """
    if not insta_data:
        return []
    
    directors = []
    
    # Try common response structures — adjust after seeing real response
    # Possibility 1: data['directors'] or data['Directors']
    # Possibility 2: data['company']['directors']
    # Possibility 3: nested in data['companyMasterData']['directors']
    
    # We'll print the full structure first time to figure out paths
    print("\n👥 Attempting to extract directors...")
    print("   (Check raw response above and adjust key paths if needed)")
    
    # Generic search for director-like keys
    def find_directors_recursive(obj, depth=0):
        if depth > 5:
            return
        if isinstance(obj, dict):
            for key, val in obj.items():
                if 'director' in key.lower() or 'din' in key.lower():
                    print(f"   Found key: {key} (type: {type(val).__name__})")
                    if isinstance(val, list) and len(val) > 0:
                        print(f"   → Contains {len(val)} items. First item keys: {list(val[0].keys()) if isinstance(val[0], dict) else val[0]}")
                if isinstance(val, (dict, list)):
                    find_directors_recursive(val, depth + 1)
        elif isinstance(obj, list):
            for item in obj[:2]:  # check first 2 items
                find_directors_recursive(item, depth + 1)
    
    find_directors_recursive(insta_data)
    return directors


# === TEST ===
print("=" * 70)
print("TEST 1: InstaFinancials — CIN → Company + Directors")
print("=" * 70)

test_cin = TEST_CINS[2]  # InstaFinancials own CIN — should definitely work
result = instafinancials_get_company(test_cin)

if result:
    print("\n📄 Full response (first 2000 chars):")
    print(json.dumps(result, indent=2, default=str)[:2000])
    
    directors = extract_directors(result)
    
    print("\n✅ InstaFinancials test complete.")
    print("   Next step: Use director names + company domain for email discovery.")
else:
    print("\n⚠️  No data returned. Check your API key.")

---
## 2. Hunter.io — Domain → Email Pattern + Emails (FREE, 25-50/month)

**What we're testing:**
- Input: Domain name
- Expected output: Email addresses, patterns, names, positions
- API: `https://api.hunter.io/v2/domain-search?domain=X&api_key=Y`
- Free tier: 25 searches/month, API accessible
- Credit cost: 1 credit per search (returns up to 10 emails)

**Key advantage:** Preview count before spending credits.

**Sign up at:** https://hunter.io/users/sign_up

In [ ]:
def hunter_domain_search(domain, limit=10):
    """
    Hunter.io Domain Search — Find all emails for a domain.
    
    FREE tier: 25 searches/month. 1 credit = up to 10 emails.
    Returns: emails with names, positions, confidence scores, email pattern.
    """
    api_key = CONFIG["HUNTER_API_KEY"]
    if not api_key:
        print("❌ HUNTER_API_KEY not set. Sign up at https://hunter.io/users/sign_up")
        return None
    
    url = "https://api.hunter.io/v2/domain-search"
    params = {
        "domain": domain,
        "api_key": api_key,
        "limit": limit,
    }
    
    print(f"\n🔍 Hunter.io: Searching domain = {domain}")
    data = safe_request("GET", url, params=params)
    
    if data and "data" in data:
        d = data["data"]
        print(f"\n📊 Results for {domain}:")
        print(f"   Organization: {d.get('organization', 'N/A')}")
        print(f"   Email pattern: {d.get('pattern', 'N/A')}")
        print(f"   Total emails found: {len(d.get('emails', []))}")
        
        for email_obj in d.get("emails", [])[:5]:
            print(f"   📧 {email_obj.get('value', '?')} "
                  f"({email_obj.get('first_name', '')} {email_obj.get('last_name', '')}) "
                  f"— {email_obj.get('position', 'N/A')} "
                  f"[confidence: {email_obj.get('confidence', '?')}%]")
    
    return data


def hunter_email_count(domain):
    """
    Hunter.io Email Count — Check how many emails exist for a domain WITHOUT spending credits.
    This is FREE and doesn't consume any credits!
    """
    api_key = CONFIG["HUNTER_API_KEY"]
    if not api_key:
        print("❌ HUNTER_API_KEY not set.")
        return None
    
    url = "https://api.hunter.io/v2/email-count"
    params = {"domain": domain}
    
    print(f"\n🔍 Hunter.io: Email count (FREE, no credits) for {domain}")
    data = safe_request("GET", url, params=params)
    
    if data and "data" in data:
        total = data["data"].get("total", 0)
        print(f"   📊 Total emails indexed: {total}")
        if total == 0:
            print(f"   ⚠️  No emails found — skip this domain to save credits.")
        else:
            print(f"   ✅ Emails exist! Worth spending 1 credit on domain search.")
    
    return data


def hunter_email_finder(domain, first_name, last_name):
    """
    Hunter.io Email Finder — Find a specific person's email.
    
    Costs 1 credit only if an email is found.
    Input: domain + first_name + last_name (from InstaFinancials directors)
    """
    api_key = CONFIG["HUNTER_API_KEY"]
    if not api_key:
        print("❌ HUNTER_API_KEY not set.")
        return None
    
    url = "https://api.hunter.io/v2/email-finder"
    params = {
        "domain": domain,
        "first_name": first_name,
        "last_name": last_name,
        "api_key": api_key,
    }
    
    print(f"\n🔍 Hunter.io: Finding email for {first_name} {last_name} @ {domain}")
    data = safe_request("GET", url, params=params)
    
    if data and "data" in data:
        d = data["data"]
        email = d.get("email", "Not found")
        score = d.get("score", 0)
        print(f"   📧 {email} [confidence: {score}%]")
    
    return data


# === TEST ===
print("=" * 70)
print("TEST 2: Hunter.io — Domain → Emails")
print("=" * 70)

# Step 1: Check email count first (FREE — no credits consumed)
for domain in TEST_DOMAINS:
    hunter_email_count(domain)

# Step 2: Only do full search on domains with results (costs 1 credit each)
# Uncomment to test — will consume credits!
# hunter_domain_search("instafinancials.com")

print("\n✅ Hunter.io test complete.")
print("   Tip: Always check email_count first (free) before domain_search (1 credit).")

---
## 3. Snov.io — Domain → Emails (FREE, 50 credits/month)

**What we're testing:**
- Auth: OAuth2 (client_id + client_secret → access_token)
- Domain search: POST to get emails for a domain
- Email finder: Find specific person's email by name + domain
- Free tier: 50 credits/month, renews monthly

**Sign up at:** https://app.snov.io/register

In [ ]:
def snov_get_token():
    """Get Snov.io access token via OAuth2."""
    client_id = CONFIG["SNOV_CLIENT_ID"]
    client_secret = CONFIG["SNOV_CLIENT_SECRET"]
    if not client_id or not client_secret:
        print("❌ SNOV_CLIENT_ID / SNOV_CLIENT_SECRET not set.")
        print("   Sign up at https://app.snov.io/register")
        print("   Find API creds at: https://app.snov.io/api-settings")
        return None
    
    url = "https://api.snov.io/v1/oauth/access_token"
    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }
    
    print("🔑 Snov.io: Getting access token...")
    data = safe_request("POST", url, json=payload)
    
    if data and "access_token" in data:
        print("   ✅ Token obtained.")
        return data["access_token"]
    return None


def snov_domain_search(domain, token, limit=10):
    """
    Snov.io Domain Search — Find emails for a domain.
    
    Costs 1 credit per 10 emails found.
    """
    url = "https://api.snov.io/v2/domain-emails-with-info"
    payload = {
        "access_token": token,
        "domain": domain,
        "type": "all",
        "limit": limit,
    }
    
    print(f"\n🔍 Snov.io: Searching domain = {domain}")
    data = safe_request("POST", url, json=payload)
    
    if data:
        emails = data.get("emails", data.get("data", {}).get("emails", []))
        print(f"   📊 Emails found: {len(emails) if isinstance(emails, list) else 'check structure'}")
        
        if isinstance(emails, list):
            for e in emails[:5]:
                if isinstance(e, dict):
                    print(f"   📧 {e.get('email', '?')} "
                          f"({e.get('firstName', '')} {e.get('lastName', '')}) "
                          f"— {e.get('position', 'N/A')}")
                else:
                    print(f"   📧 {e}")
    
    return data


def snov_email_finder(domain, first_name, last_name, token):
    """
    Snov.io Email Finder — Find a specific person's email.
    
    Costs 1 credit if found.
    """
    url = "https://api.snov.io/v1/get-emails-from-names"
    payload = {
        "access_token": token,
        "firstName": first_name,
        "lastName": last_name,
        "domain": domain,
    }
    
    print(f"\n🔍 Snov.io: Finding email for {first_name} {last_name} @ {domain}")
    data = safe_request("POST", url, json=payload)
    
    if data:
        email_data = data.get("data", data)
        if isinstance(email_data, dict):
            emails = email_data.get("emails", [])
            for e in emails:
                if isinstance(e, dict):
                    print(f"   📧 {e.get('email', '?')} [status: {e.get('emailStatus', '?')}]")
    
    return data


# === TEST ===
print("=" * 70)
print("TEST 3: Snov.io — Domain → Emails")
print("=" * 70)

snov_token = snov_get_token()

if snov_token:
    # Domain search (costs credits — uncomment to test)
    # snov_domain_search("instafinancials.com", snov_token)
    
    # Email finder (costs credits — uncomment to test)
    # snov_email_finder("flipkart.com", "Kalyan", "Krishnamurthy", snov_token)
    print("\n   Token ready. Uncomment search calls above to test (will consume credits).")

print("\n✅ Snov.io test complete.")

---
## 4. Prospeo — Name + Domain → Email (FREE, 75/month)

**What we're testing:**
- Email finder: name + domain → verified email
- No credit charged if email not found (individual mode)
- Free tier: 75 email credits/month

**Sign up at:** https://app.prospeo.io/register

**API docs:** https://prospeo.io/api/email-finder

In [ ]:
def prospeo_email_finder(domain, full_name=None, first_name=None, last_name=None):
    """
    Prospeo Email Finder — Find email by name + domain.
    
    FREE tier: 75 credits/month. No charge if email not found.
    Accepts either full_name OR first_name + last_name.
    """
    api_key = CONFIG["PROSPEO_API_KEY"]
    if not api_key:
        print("❌ PROSPEO_API_KEY not set. Sign up at https://app.prospeo.io/register")
        print("   Find API key at: https://app.prospeo.io/api")
        return None
    
    url = "https://api.prospeo.io/email-finder"
    headers = {
        "Content-Type": "application/json",
        "X-KEY": api_key,
    }
    payload = {"company": domain}
    
    if full_name:
        payload["full_name"] = full_name
    else:
        payload["first_name"] = first_name
        payload["last_name"] = last_name
    
    name_str = full_name or f"{first_name} {last_name}"
    print(f"\n🔍 Prospeo: Finding email for {name_str} @ {domain}")
    data = safe_request("POST", url, headers=headers, json=payload)
    
    if data:
        response = data.get("response", data)
        if isinstance(response, dict):
            email = response.get("email", "Not found")
            confidence = response.get("email_status", response.get("confidence", "?"))
            print(f"   📧 {email} [status: {confidence}]")
    
    return data


def prospeo_domain_search(domain):
    """
    Prospeo Domain Search — Find all emails for a domain.
    """
    api_key = CONFIG["PROSPEO_API_KEY"]
    if not api_key:
        print("❌ PROSPEO_API_KEY not set.")
        return None
    
    url = "https://api.prospeo.io/domain-search"
    headers = {
        "Content-Type": "application/json",
        "X-KEY": api_key,
    }
    payload = {"company": domain, "limit": 10}
    
    print(f"\n🔍 Prospeo: Domain search for {domain}")
    data = safe_request("POST", url, headers=headers, json=payload)
    
    if data:
        response = data.get("response", data)
        if isinstance(response, dict):
            emails = response.get("email_list", response.get("emails", []))
            print(f"   📊 Emails found: {len(emails) if isinstance(emails, list) else 'check structure'}")
            if isinstance(emails, list):
                for e in emails[:5]:
                    if isinstance(e, dict):
                        print(f"   📧 {e.get('email', '?')} — {e.get('first_name', '')} {e.get('last_name', '')}")
    
    return data


# === TEST ===
print("=" * 70)
print("TEST 4: Prospeo — Name + Domain → Email")
print("=" * 70)

# Uncomment to test (costs credits if email found):
# prospeo_email_finder("flipkart.com", first_name="Kalyan", last_name="Krishnamurthy")
# prospeo_domain_search("instafinancials.com")

print("\n   Uncomment the test calls above to try (will consume credits if email found).")
print("✅ Prospeo test complete.")

---
## 5. Apollo.io — Company Search → Contacts (FREE, 10k views)

**What we're testing:**
- People search by company name, title, location
- Free tier: Unlimited views, 10 exports/month
- API: May not be available on free tier — testing to confirm

**Sign up at:** https://www.apollo.io/sign-up (use corporate email!)

In [ ]:
def apollo_people_search(company_name, titles=None, limit=5):
    """
    Apollo.io People Search — Find contacts at a company.
    
    FREE tier: Views are free, but API access may require paid plan.
    Testing to confirm.
    """
    api_key = CONFIG["APOLLO_API_KEY"]
    if not api_key:
        print("❌ APOLLO_API_KEY not set.")
        print("   Sign up at https://www.apollo.io/sign-up")
        print("   Find API key at: Settings → Integrations → API")
        return None
    
    url = "https://api.apollo.io/v1/mixed_people/search"
    headers = {
        "Content-Type": "application/json",
        "Cache-Control": "no-cache",
    }
    payload = {
        "api_key": api_key,
        "q_organization_name": company_name,
        "page": 1,
        "per_page": limit,
    }
    
    if titles:
        payload["person_titles"] = titles  # e.g. ["Director", "CEO", "Marketing Head"]
    
    print(f"\n🔍 Apollo.io: Searching people at '{company_name}'")
    data = safe_request("POST", url, headers=headers, json=payload)
    
    if data:
        people = data.get("people", [])
        print(f"   📊 People found: {len(people)}")
        
        for p in people[:5]:
            name = f"{p.get('first_name', '')} {p.get('last_name', '')}".strip()
            title = p.get("title", "N/A")
            email = p.get("email", "hidden/not available")
            org = p.get("organization", {}).get("name", "N/A") if isinstance(p.get("organization"), dict) else "N/A"
            print(f"   👤 {name} — {title} @ {org}")
            print(f"      📧 {email}")
    
    return data


# === TEST ===
print("=" * 70)
print("TEST 5: Apollo.io — Company Search → People")
print("=" * 70)

# Uncomment to test:
# apollo_people_search("Flipkart", titles=["Director", "CEO", "VP"])

print("\n   Uncomment the test call above to try.")
print("   Note: API may not work on free tier — if 403/401, use web UI instead.")
print("✅ Apollo.io test complete.")

---
## 6. Lusha — Company → Contacts (FREE, 40-70 credits/month)

**API NOT available on free tier** (Scale plan only).

Usage on free tier is **Chrome extension + web app only**.

Keeping this cell as a placeholder — if you ever upgrade or find an API workaround.

In [ ]:
def lusha_person_lookup(first_name, last_name, company_name):
    """
    Lusha Person API — NOT available on free tier.
    
    Requires Scale plan (custom enterprise pricing).
    Keeping as placeholder for future use.
    
    On free tier: Use web UI at https://app.lusha.com or Chrome extension.
    - Search is FREE (doesn't consume credits)
    - Revealing email costs 1 credit
    - Revealing phone costs 10 credits (was 5, increased in 2026)
    - 40-70 credits/month on free tier
    """
    api_key = CONFIG["LUSHA_API_KEY"]
    if not api_key:
        print("⚠️  Lusha API requires Scale plan (paid). Skipping API test.")
        print("   Use the Lusha web app or Chrome extension for free tier.")
        print("   Web app: https://app.lusha.com")
        print("   Tip: Search is free. Only 'Reveal' costs credits.")
        return None
    
    # API endpoint (for Scale plan users)
    url = "https://api.lusha.com/person"
    headers = {
        "api_key": api_key,
        "Content-Type": "application/json",
    }
    params = {
        "firstName": first_name,
        "lastName": last_name,
        "company": company_name,
    }
    
    print(f"\n🔍 Lusha: Looking up {first_name} {last_name} @ {company_name}")
    data = safe_request("GET", url, headers=headers, params=params)
    
    return data


# === TEST ===
print("=" * 70)
print("TEST 6: Lusha — Person Lookup (API requires paid plan)")
print("=" * 70)

lusha_person_lookup("Kalyan", "Krishnamurthy", "Flipkart")

print("\n✅ Lusha test complete.")
print("   Action: Use Lusha web app manually for 40-70 free reveals/month.")

---
## 7. Lix — LinkedIn → Verified Emails (FREE, 50/month)

**Lix is Chrome extension only on free tier.**

They do have an API (https://developer.lix-it.com/) but it may require a paid plan.

Keeping as placeholder — the main workflow is:
1. Search LinkedIn for `"Marketing Director" at "Company Name"`
2. Use Lix Chrome extension to export with verified emails
3. Get 50 verified emails/month for free

In [ ]:
def lix_email_finder(linkedin_url=None):
    """
    Lix API — Find email from LinkedIn profile URL.
    
    Free tier: 50 verified emails/month via Chrome extension.
    API may require paid plan — testing.
    """
    api_key = CONFIG["LIX_API_KEY"]
    if not api_key:
        print("⚠️  LIX_API_KEY not set. Lix free tier is Chrome extension based.")
        print("   Install: https://chrome.google.com/webstore/detail/lix-b2b-email-finder/ceplokfhfeekddamgoaojabdhkggafnk")
        print("   Free: 50 verified emails/month from LinkedIn profiles.")
        print("\n   Manual workflow:")
        print("   1. Go to LinkedIn → Search 'Marketing Director at [Company]'")
        print("   2. Click Lix extension → Export with emails")
        print("   3. Download CSV with name + verified email")
        return None
    
    # Lix API endpoint (if API key available)
    url = "https://api.lix-it.com/v1/person"
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {"profile_link": linkedin_url} if linkedin_url else {}
    
    print(f"\n🔍 Lix: Looking up {linkedin_url or 'test'}")
    data = safe_request("GET", url, headers=headers, params=params)
    
    return data


# === TEST ===
print("=" * 70)
print("TEST 7: Lix — LinkedIn → Verified Emails")
print("=" * 70)

lix_email_finder()

print("\n✅ Lix test complete.")
print("   Action: Use Lix Chrome extension on LinkedIn for 50 free emails/month.")

---
## Summary: What Works via API vs Manual

| Provider | API on Free? | Monthly Free | Primary Use |
|---|---|---|---|
| **InstaFinancials** | ✅ YES | Unlimited | CIN → Director names + DINs |
| **Hunter.io** | ✅ YES | 25 searches + 50 verifications | Domain → email pattern + emails |
| **Snov.io** | ✅ YES | 50 credits | Domain → emails, name → email |
| **Prospeo** | ✅ YES (check) | 75 emails | Name + domain → verified email |
| **Apollo.io** | ⚠️ Maybe | 10k views, 10 exports | Company → people with emails |
| **Lusha** | ❌ NO | 40-70 credits | Manual via web app/extension |
| **Lix** | ❌ NO | 50 emails | Manual via Chrome extension on LinkedIn |

---
## Next Steps

1. **Sign up** for each provider and paste API keys in the CONFIG cell
2. **Run InstaFinancials first** (free, unlimited) to get director names
3. **Run Hunter email_count** (free) to check which domains have indexed emails
4. **Then spend credits wisely** on Hunter domain_search, Snov.io, or Prospeo
5. **Use Lusha + Lix manually** via Chrome extensions for additional 90-120 emails/month

Once all providers are validated, we build the automated pipeline script.